# 61 · Seven ways to find pathways that differ between human and mouse proximal tubule

**The question.** Which pathways differ between the human and the mouse proximal tubule (PT), and
does the difference depend on where along the tubule one looks? All seven methods answer this from the
same data:
- 12,272 PT cross-sections from 2 control mice and 2 cortex sections of 1 human donor;
- 11,071 genes measured in both species;
- 1,513 Reactome, Hallmark and KEGG pathways.

**The seven methods:**

| # | Method | Code | What it compares |
|---|---|---|---|
| 1 | **Whole PT**: pseudobulk DE + signed GSEA | A1·GSEA | the average over the whole PT |
| 2 | **Discrete**: S1/S2/S3 segment steps, joint test | A4·JNT | the profile over three zones; needs no coordinate |
| 3 | **Continuous, pathway-first, rank-based**: T_spatial, joint test (primary) | A6·JNT | the profile along the coordinate |
| 4 | **Continuous, pathway-first, threshold-based**: ORA on split-plot significant genes | A6·ORAc | the same, through a list of significant genes |
| 7 | **Continuous, pathway-first, rank-based with GSEA**: preranked GSEA on T_spatial | A6·GSEA | the same ranking as method 3, with GSEA's null |
| 5 | **Continuous, gene-first, naive**: curve modules of the top 10% T_spatial genes + ORA | B1·T10 | groups of genes with the same difference shape |
| 6 | **Continuous, gene-first, specimen-aware**: curve modules of the split-plot significant genes + ORA | B2·split | the same, from a calibrated gene list |

Method 7 was added last, so it keeps its number but is shown next to methods 3 and 4, the other two
continuous pathway-first methods.

The continuous methods form a 2 × 2:

| | gene list from structure-level T_spatial ranks | gene list from the specimen-level split-plot test |
|---|---|---|
| **pathway-first** (test each pathway, then look at its genes) | 3 · A6·JNT and 7 · A6·GSEA (all genes, ranked) | 4 · A6·ORAc |
| **gene-first** (group the genes, then name the groups) | 5 · B1·T10 (top 10%) | 6 · B2·split |

- Methods 3 and 7 use the same ranking and differ only in the set test.
- Methods 4 and 6 use the same gene list: the genes whose species × position interaction is
  significant against specimen-to-specimen variation.

**How every method is judged.**
- **The species split** puts the two human sections against the two mice.
- **Each relabeling** pairs one human section with one mouse, which removes the species contrast
  and leaves only specimen differences. A method is **specific** when the relabelings call few
  pathways: mean relabeled calls ÷ species calls (NCDR) below 0.25, and each relabeling below 5% of
  tested pathways.
- **Specific methods are replicated externally.** Their calls are checked in two independent human
  cortex atlases (Census, Lake/KPMP) against male mice.

**Notebook 63** holds the full grid of 48 methods behind the supplementary figure.

**Assumptions.** SCF13 is treated as an exact coordinate. Specimens, not structures, are the
replicates, and all specimens are male. Results are descriptive. Every method is computed or loaded
exactly as in notebook 63 and checked against its recorded values.

## 1 · Setup

In [ ]:
import argparse
import glob
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
nb12 = results_root / 'pt_pathway_remodeling_scfates'
nb31 = results_root / 'pt_pathway_method_selection'
nb37 = results_root / 'pt_pathway_final' / 'scfates'
nb45 = results_root / 'pt_revision_method' / 'tables'
nb45_cache = results_root / 'pt_revision_method' / 'stage_cache'
nb50 = results_root / 'pt_primary_pathways' / 'tables'
nb50_cache = results_root / 'pt_primary_pathways' / 'stage_cache'
nb56 = results_root / 'pt_pathway_continuous' / '56_taxonomy'
nb57 = results_root / 'pt_pathway_continuous' / '57_beyond_steps' / 'scf13' / 'tables'
external = data_root / 'external'
output = results_root / 'pt_pathway_methods_compared'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)
REQUIRED = [results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv',
            nb31 / 'conventional_deseq2_all_partitions.csv', nb31 / 'conventional_signed_gsea_all_partitions.csv',
            nb12 / 'gene_statistics.csv', results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv',
            nb37 / 'tables' / 'matched_and_joint_tests_all_partitions.csv', nb37 / 'tables' / 'table_S1_strategy_specificity.csv',
            nb37 / 'tables' / 'table_S2_pathways.csv', nb37 / 'tables' / 'gene_level_statistics.csv',
            nb37 / 'tables' / 'segment_transitions_by_specimen.csv',
            nb45 / 'anchor_our_cohort.csv', nb50 / 'cohort_step_screen_per_pathway.csv', nb50 / 'primary_pathway_list.csv',
            nb56 / 'tables' / 'f_taxonomy.csv', nb56 / 'tables' / 'f_species_calls_by_approach.csv',
            nb57 / 'per_pathway_beyond_steps.csv',
            results_root / 'pt_revision_classes' / 'reviewed' / 'tables' / 'gene_classes_symmetric.csv',
            external / 'census_pt_segments' / 'human_pt_segment_pseudobulk_counts.csv.gz',
            external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz']
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 12, 31, 37, 43, 45, 50, 56 and 57 first)')

NOTEBOOK_LOGIC_VERSION = '61.seven_methods.1'  # Bump when a cached calculation changes.
ALPHA, NCDR_MAX, NULL_RATE = .05, .25, .05
LFC = 1.0                     # DE genes need |log2FC| >= 1 (notebook 12); used only by the worked-example rules
K_MODULES, KM_STARTS, KM_SEED, TOP_SHARE = 8, 20, 56, .10   # notebook 56
K_RANGE, N_RESAMPLE = range(3, 9), 20                       # module-count selection (notebook 56)
N_PERM_EXTERNAL, MIN_CALLS_EXTERNAL, MIN_CELLS_EXTERNAL = 999, 5, 50
EXTERNAL_SEED, EXTERNAL_CHUNKS = 50, 27         # 27 chunks of 37 permutations, each seeded by its own id (notebook 63)
EXAMPLE_BINS = 12             # specimen x coordinate-bin means in the worked examples (notebook 41)
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '32'))  # parallel workers; never part of a cache key

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from joblib import Parallel, delayed, parallel_config
from scipy import sparse
from scipy.stats import hypergeom, norm
from statsmodels.stats.multitest import multipletests
from threadpoolctl import threadpool_limits

MM = 1 / 25.4
HUMAN, MOUSE, NULL, LIGHT, INK, MUTED, ACCENT = '#D55E00', '#0072B2', '#9A9A9A', '#D9D9D9', '#222222', '#666666', '#009E73'
SPECIMENS = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
SPECIMEN_MARKER = {'Ctrl1A2': 'o', 'Ctrl1A4': 's', 'HUK1_COR1': 'o', 'HUK1_MED1': 's'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 80)
print('Results folder:', output, '· parallel workers:', N_JOBS)

## 2 · Structures, genes, pathways, partitions and shared helpers

The construction is notebook 37's on SCF13:
- the structures inside every specimen's 1st–99th percentile range of the coordinate;
- the genes measured in both species and detected in ≥ 2% of structures;
- the pathways with 10–300 tested genes.

In [ ]:
from pseudospace.coordinate_inputs import pt_inputs, read_coordinate
from pseudospace.pathway_calibration import balanced_partitions
from pseudospace.pathway_decoys import joint_matched_test
from pseudospace.stage_cache import cached_frame, cached_payload, digest
from pseudospace.stats_gam import gam_internal_knots, make_gam_design

inputs = pt_inputs(results_root, data_root, read_coordinate(results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv'))
position, human, specimen, segment = (inputs[k] for k in ('position', 'human', 'specimen', 'segment'))
genes, gene_sets, strata = inputs['genes'], inputs['gene_sets'], inputs['strata']
pathways = list(gene_sets)
partitions = balanced_partitions(specimen, np.where(human, 'human', 'mouse'))
SWAPS = [p for p in partitions if p != 'species']
PARTS = ['species', *SWAPS]
ALL_SPECIMENS = SPECIMENS['mouse'] + SPECIMENS['human']
assert len(position) == 12272 and len(genes) == 11071 and len(pathways) == 1513, 'notebook 37 universe changed'
table37 = pd.read_csv(nb37 / 'tables' / 'table_S2_pathways.csv').set_index('pathway_id')
NAMES = table37.name.reindex(pathways)
VIF = table37.vif.reindex(pathways)
spec37 = pd.read_csv(nb37 / 'tables' / 'table_S1_strategy_specificity.csv').set_index('strategy')
gene37 = pd.read_csv(nb37 / 'tables' / 'gene_level_statistics.csv', index_col=0).T_spatial.reindex(genes)
cache = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION])
input_key = {'Y': digest(inputs['Y']), 'position': position, 'specimen': specimen, 'genes': genes}
grid = inputs['grid']
lo, hi = grid[0], grid[-1]
knots = gam_internal_knots(position, basis_df=6)
print('Partitions:', partitions)

CALLS, EVIDENCE, METHODS = {}, {}, []
RETURNS = ['yes/no', 'direction', 'location', 'shape', 'genes']
LOADED = 'loaded'


def register(key, number, label, calls, *, evidence=None, returns=('yes/no',), runtime=None, note=''):
    """Record one method: its calls under the species split and both relabelings, and the specificity rule."""
    CALLS[key] = {p: set(calls[p]) for p in PARTS}
    if evidence is not None:
        EVIDENCE[key] = pd.Series(evidence, dtype=float).reindex(pathways)
    relabels = [len(CALLS[key][s]) for s in SWAPS]
    n_species = len(CALLS[key]['species'])
    ncdr = np.mean(relabels) / n_species if n_species else 0.
    passes = bool(n_species and ncdr < NCDR_MAX and max(relabels) < NULL_RATE * len(pathways))
    METHODS.append({'key': key, '#': number, 'method': label, 'species calls': n_species,
                    'relabeled calls': ', '.join(map(str, relabels)), 'NCDR': ncdr, 'specific': passes,
                    'runtime (s)': runtime, **{r: r in returns for r in RETURNS}, 'note': note})
    print(f"{number} · {key:<9} {n_species:>4} species calls; relabeled {relabels}; {'passes' if passes else 'fails'} the relabeling rule")


def timed(stage, compute, kind='frame', **cache_kwargs):
    """Cache a single (serial) computation together with its uncached wall time; returns (result, seconds)."""
    def wrapped():
        t0 = time.perf_counter()
        out = compute()
        seconds = time.perf_counter() - t0
        if kind == 'frame':
            return out.assign(__seconds=seconds)
        out = dict(out)
        out['__seconds'] = np.array(seconds)
        return out
    if kind == 'frame':
        result = cached_frame(stage, wrapped, root=cache, code=code, **cache_kwargs)
        return result.drop(columns='__seconds'), float(result['__seconds'].iloc[0])
    result = cached_payload(stage, wrapped, root=cache, code=code, **cache_kwargs)
    return {k: v for k, v in result.items() if k != '__seconds'}, float(result['__seconds'])


def _run_task(function, args):
    t0 = time.perf_counter()
    with threadpool_limits(1):
        out = function(*args)
    return out, time.perf_counter() - t0


def fan_out(function, task_args, n_jobs=None):
    """``function(*args)`` for every args tuple, in task order, on ``n_jobs`` loky workers (one BLAS thread each).

    Random tasks carry their own seeds, derived from fixed task ids, so results do not depend on ``n_jobs``.
    """
    jobs = N_JOBS if n_jobs is None else n_jobs
    if jobs == 1 or len(task_args) == 1:
        pairs = [_run_task(function, a) for a in task_args]
    else:
        with parallel_config(backend='loky', inner_max_num_threads=1):
            pairs = Parallel(n_jobs=min(jobs, len(task_args)))(delayed(_run_task)(function, a) for a in task_args)
    return [p[0] for p in pairs], [p[1] for p in pairs]


def cached_tasks(stage, function, tasks, **cache_kwargs):
    """Fan out ``tasks`` (task id -> args) and cache all results as one stage-cache entry (key never includes n_jobs).

    Returns {task id: (result dict, seconds)}; seconds are each task's own compute time.
    """
    ids = list(tasks)

    def compute():
        results, seconds = fan_out(function, [tasks[i] for i in ids])
        payload = {}
        for i, result, sec in zip(ids, results, seconds):
            payload.update({f'{i}|{k}': np.asarray(v) for k, v in result.items()})
            payload[f'{i}|__seconds'] = np.array(sec)
        return payload
    stored = cached_payload(stage, compute, root=cache, code=code, **cache_kwargs)
    out = {i: {} for i in ids}
    for key, value in stored.items():
        i, k = key.rsplit('|', 1)
        out[i][k] = value
    return {i: (out[i], float(out[i].pop('__seconds'))) for i in ids}


def slug(p):
    return p.replace(':', '_').replace('+', '_')


def membership(sets, universe):
    index = pd.Index(universe)
    rows, cols = [], []
    for i, members in enumerate(sets.values()):
        idx = index.get_indexer(members)
        idx = idx[idx >= 0]
        rows.extend([i] * len(idx)), cols.extend(idx)
    return sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(sets), len(index)))


member_matrix = membership(gene_sets, genes)


def ora_pvalues(selected, matrix):
    """One-sided hypergeometric over-representation p for a boolean gene selection."""
    selected = np.asarray(selected, bool)
    if not selected.any():
        return np.ones(matrix.shape[0])
    k = np.asarray(matrix @ selected.astype(float)).ravel()
    return hypergeom.sf(k - 1, len(selected), np.asarray(matrix.sum(axis=1)).ravel(), selected.sum())


def bh(p):
    return multipletests(np.nan_to_num(np.asarray(p, float), nan=1.), method='fdr_bh')[1]


def newest(folder, stage, kind='npz'):
    files = glob.glob(str(folder / f'{stage}__*.{kind}'))
    if not files:
        raise FileNotFoundError(f'{stage} not found in {folder}')
    files.sort(key=lambda p: json.loads(Path(p + '.key.json').read_text())['written_utc'])
    if kind == 'csv':
        return pd.read_csv(files[-1])
    with np.load(files[-1]) as stored:
        return {k: stored[k] for k in stored.files}


def check_counts(key, strategy):
    row = spec37.loc[strategy]
    assert [len(CALLS[key][p]) for p in PARTS] == [row.species_calls, row.relabel_calls_Ctrl1A2, row.relabel_calls_Ctrl1A4], \
        (key, [len(CALLS[key][p]) for p in PARTS])

## 3 · The seven methods

### Method 1 · Whole PT: pseudobulk DE + signed GSEA (A1·GSEA)

**What it does.**
1. Each specimen's counts are summed over the whole PT.
2. DESeq2 tests every gene, human against mouse.
3. Genes are ranked by their Wald statistic, and preranked GSEA asks whether a pathway's genes
   gather at the top or the bottom of the ranking.

This is the analysis most papers run.

**One call means:** averaged over the whole PT, this pathway's genes are shifted between species
more than other genes are, in the direction given by the sign of NES.

**What it cannot tell you:**
- where along the tubule the difference is;
- whether a difference is real or specimen-to-specimen. With two specimens per species, GSEA's gene
  permutation null ignores that specimens differ, and the relabeling check below shows the cost.

The statistics are notebook 31's, loaded and checked against notebook 37's counts.

In [ ]:
deseq = pd.read_csv(nb31 / 'conventional_deseq2_all_partitions.csv')
gsea31 = pd.read_csv(nb31 / 'conventional_signed_gsea_all_partitions.csv')
whole = gsea31[gsea31.family.eq('whole_PT')]
whole_species = whole[whole.partition.eq('species')].set_index('pathway_id')
register('A1·GSEA', 1, 'Whole PT: pseudobulk DE + signed GSEA', {p: whole.pathway_id[whole.partition.eq(p) & whole.q.le(ALPHA)] for p in PARTS},
         evidence=-np.log10(whole_species.p.clip(lower=1e-300)), returns=('yes/no', 'direction', 'genes'), runtime=LOADED)
check_counts('A1·GSEA', '1 · Whole-PT pseudobulk + signed GSEA')
WHOLE_DIRECTION = np.sign(whole_species.NES)

### Method 2 · Discrete: S1/S2/S3 segment steps with the joint test (A4·JNT)

**What it does.**
- For each gene, it asks whether the step pattern from S1 to S2 to S3 differs between species,
  beyond a species offset. The statistic is the partial F of the species × segment terms, with
  shared segment intercepts, computed on the structures.
- It then asks whether a pathway's genes rank higher than genes matched on expression, detection
  and coverage. The z-score is divided by √VIF, which allows for correlation between the pathway's
  genes. This is the **joint test**, the same set test as method 3.
- Position enters only as three zones, so no coordinate is needed.

**One call means:** the pathway's S1 → S2 → S3 profile differs in shape between species more than
matched genes do.

**What it cannot tell you:**
- anything inside a segment;
- an average offset, which the model removes on purpose.

The statistic is computed exactly from notebook 45's cell summaries and checked against notebooks 45
and 50.

In [ ]:
from pseudospace.pathways import build_pathway_membership
from pseudospace.specimen_null import matched_auc_null, summary_partial_f

pool = newest(nb45_cache, 'pool_our_cohort')
assert list(pool['genes'].astype(str)) == list(genes), 'notebook 45 pool genes differ from the universe'
universe12 = pd.read_csv(nb12 / 'gene_statistics.csv', index_col=0).index.tolist()
coverage45 = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json').read_text()), universe12,
    ortholog_map=pd.read_csv(results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv'), library_name=lib,
    tested=universe12, min_genes=1) for lib in ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')], ignore_index=True)
coverage45['pathway_id'] = coverage45.library + '::' + coverage45.pathway
coverage45 = coverage45[coverage45.n_tested.between(10, min(300, len(universe12) - 1))]
pool_genes = set(pool['genes'].astype(str))
sets45 = {row.pathway_id: [g for g in row.genes_present if g in pool_genes] for row in coverage45.itertuples()}
sets45 = {p: sets45[p] for p in pool['pathways'].astype(str)}   # notebook 45's order and members
pool_vif = pd.Series(pool['vif'], index=list(sets45))
pool_donors = list(pool['donors'].astype(str))
species_of = {d: float(d in SPECIMENS['human']) for d in pool_donors}
designs45 = {'species': (species_of, None)}
designs45.update({s: ({d: float(d in partitions[s]) for d in pool_donors}, species_of) for s in SWAPS})


def step_statistics():
    return {p: summary_partial_f(pool, pool_donors, group, nuisance)['T_steps'] for p, (group, nuisance) in designs45.items()}


t_steps, sec_steps = timed('t_steps', step_statistics, kind='payload', inputs={'pool': digest(pool['mean'])})
step_tests = {p: matched_auc_null(pd.DataFrame({'T_steps': t_steps[p]}, index=genes), sets45, pool['strata']).set_index('pathway_id')
              for p in PARTS}
for p in PARTS:
    step_tests[p]['z_joint'] = step_tests[p].z_matched / np.sqrt(pool_vif)
register('A4·JNT', 2, 'Discrete: S1/S2/S3 segment steps, joint test',
         {p: t.index[t.effect.gt(0) & (bh(norm.sf(t.z_joint)) <= ALPHA)] for p, t in step_tests.items()},
         evidence=step_tests['species'].z_joint, returns=('yes/no', 'location'), runtime=sec_steps)
steps50 = pd.read_csv(nb50 / 'cohort_step_screen_per_pathway.csv').set_index('pathway_id')
assert CALLS['A4·JNT']['species'] == set(steps50.index[steps50.called.astype(bool)])          # Guard: notebook 50
anchor = pd.read_csv(nb45 / 'anchor_our_cohort.csv').set_index('strategy').loc['P1 T_steps joint']
assert [len(CALLS['A4·JNT'][p]) for p in PARTS] == [anchor.species, anchor.relabel_Ctrl1A2, anchor.relabel_Ctrl1A4]   # Guard: notebook 45

### Method 3 · Continuous, pathway-first, rank-based: T_spatial with the joint test (A6·JNT, primary)

**What it does.**
- For each gene, a 6-df spline along the coordinate is fitted in each species, and **T_spatial**
  is the partial F of the species × position terms beyond a species offset. Each species gets half
  the weight and each specimen an equal share.
- All genes are ranked by T_spatial, and the pathway is tested with the same joint test as method 2.

**One call means:** the shape of the pathway genes' expression along the PT depends on species more
than for matched genes.

**It also returns:**
- the pathway's difference curve Δ(s), human minus mouse, along the coordinate;
- where Δ(s) peaks, and the direction there;
- the genes that carry the signal.

**What it cannot tell you:**
- an average offset, which is removed on purpose;
- whether a single gene is significant. T_spatial ranks genes; its degrees of freedom count
  structures, so it is not a calibrated gene test.

The results are notebook 37's, loaded and checked; the joint z is recomputed.

In [ ]:
tests37 = pd.read_csv(nb37 / 'tables' / 'matched_and_joint_tests_all_partitions.csv')
tests37 = tests37[tests37.partition.isin(PARTS)]
recomputed = joint_matched_test(tests37.drop(columns=['z_matched', 'vif', 'z_joint', 'p_joint', 'q_joint']), VIF)
np.testing.assert_allclose(recomputed.z_joint, tests37.z_joint, rtol=1e-9)   # Guard: joint z = matched z ÷ √VIF
spatial37 = tests37[tests37.statistic.eq('T_spatial')]
register('A6·JNT', 3, 'Continuous, pathway-first, rank-based: T_spatial, joint test',
         {p: spatial37.pathway_id[spatial37.partition.eq(p) & spatial37.effect.gt(0) & spatial37.q_joint.le(ALPHA)] for p in PARTS},
         evidence=spatial37[spatial37.partition.eq('species')].set_index('pathway_id').z_joint,
         returns=('yes/no', 'direction', 'location', 'shape', 'genes'), runtime=LOADED)
check_counts('A6·JNT', '8 · Gene GAM × smooth position (T_spatial) (joint test)')
curves = {p: newest(nb56 / 'stage_cache', 'curves_' + slug(p)) for p in PARTS}   # Δ(s) per gene (notebook 56)
np.testing.assert_allclose(curves['species']['T_spatial'], gene37, rtol=1e-6)    # Guard: notebook 37's T_spatial

### Method 4 · Continuous, pathway-first, threshold-based: ORA on split-plot significant genes (A6·ORAc)

**What it does.**
1. **Each gene is tested at specimen level.** This is the split plot of notebook 37:
   - expression is averaged in specimen × 8-bin cells along the coordinate;
   - the species × position interaction is judged against specimen × position variation, with a
     moderated F.
2. Genes with BH q ≤ 0.05 form the list. With the species split this is 4,588 genes; with either
   relabeling it is none.
3. ORA (a hypergeometric test) asks whether a pathway holds more listed genes than chance.

**One call means:** the pathway is over-represented among genes whose positional profile differs
between species, judged against the specimens.

**What it cannot tell you:**
- how strongly each gene differs. The list is yes or no, and the 4,588 genes weigh equally;
- anything about pathways whose genes change only modestly.

Its gene list is the same one that method 6 clusters.

In [ ]:
split37 = {p: newest(nb37 / 'stage_cache', f'split_plot_{slug(p)}') for p in PARTS}
SPLIT_GENES = {p: bh(split37[p]['p']) <= ALPHA for p in PARTS}
print('Split-plot significant genes (BH ≤ 0.05):', {p: int(v.sum()) for p, v in SPLIT_GENES.items()})
t0 = time.perf_counter()
ora_split = {p: ora_pvalues(SPLIT_GENES[p], member_matrix) for p in PARTS}
register('A6·ORAc', 4, 'Continuous, pathway-first, threshold-based: split-plot significant genes + ORA',
         {p: np.array(pathways)[bh(v) <= ALPHA] for p, v in ora_split.items()},
         evidence=-np.log10(pd.Series(ora_split['species'], index=pathways)),
         returns=('yes/no', 'direction', 'location', 'shape', 'genes'), runtime=time.perf_counter() - t0)

### Method 7 · Continuous, pathway-first, rank-based with GSEA: preranked GSEA on T_spatial (A6·GSEA)

**What it does.** It uses the same T_spatial ranking as method 3, but tests each pathway with
preranked GSEA (weight 0, multilevel), the most common set test.

**One call means:** the pathway's genes gather near the top of the T_spatial ranking more than
random gene sets of the same size do.

**Why it is less selective than the joint test.** GSEA calls about 20% of pathways where the joint
test calls 4%, and section 4 shows its calls replicate only in Lake, not in Census. The joint test
of method 3 adds two corrections that GSEA lacks:
- **Matched background.** GSEA compares a pathway with random gene sets drawn from all genes. The
  joint test draws them from genes matched on expression, detection and coverage. Highly expressed,
  well-detected genes give the spline model more power, so they rank higher on T_spatial, and
  pathways made of such genes look enriched to GSEA.
- **Gene correlation.** GSEA treats a pathway's genes as independent. Co-regulated genes move
  together, so their ranks carry less independent evidence than their number suggests. The joint
  test divides the z-score by √VIF, the variance inflation estimated from the within-specimen
  correlation of the pathway's genes.

**What it cannot tell you:** the same as method 3, and its calls cannot be ranked by an effect that
accounts for these two biases.

The results are notebook 31's, loaded and checked against notebook 37's counts.

In [ ]:
hits31 = pd.read_csv(nb31 / 'strategy_hits_all_partitions.csv')
gsea_spatial = hits31[hits31.strategy.eq('8c · T_spatial, preranked GSEA')]
spatial_gsea37 = newest(nb37 / 'stage_cache', 'spatial_gsea_species', kind='csv').set_index('pathway_id')
register('A6·GSEA', 7, 'Continuous, pathway-first, rank-based: T_spatial, preranked GSEA',
         {p: gsea_spatial.pathway_id[gsea_spatial.partition.eq(p) & gsea_spatial.hit] for p in PARTS},
         evidence=-np.log10(spatial_gsea37.p.clip(lower=1e-300)), returns=('yes/no', 'direction', 'location', 'shape', 'genes'),
         runtime=LOADED)
check_counts('A6·GSEA', '8c · T_spatial, preranked GSEA')

### Methods 5 and 6 · Continuous, gene-first: curve modules + ORA

**What they do.**
1. Take a list of genes and each gene's difference curve Δ(s), human minus mouse along the
   coordinate, centred and scaled.
2. Group the curves by shape with k-means.
3. Ask with ORA which pathways are over-represented in each group (module). A pathway is called when
   it is enriched in any module.

The two methods differ only in the gene list:
- **Method 5, naive (B1·T10):** the top 10% of genes by T_spatial, with k = 8 (notebook 56's
  stability choice).
- **Method 6, specimen-aware (B2·split):** the split-plot significant genes of method 4. k is chosen
  among 3–8 by resampling stability on the species run.

The relabelings recluster their own lists. A relabeling with fewer than 10·k significant genes has
no modules and no calls.

**One call means:** the pathway's genes are over-represented in a group of genes that share one
shape of species difference.

**What they cannot tell you:**
- whether the pathway as a whole differs. A pathway whose genes change in two different ways is
  split across modules;
- anything beyond the list. ORA here is not matched on expression, detection or coverage.

**What they can show:** a coherent gene group that no library pathway describes (section 5).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score


def standardize(d):
    return (d - d.mean(axis=1, keepdims=True)) / np.maximum(d.std(axis=1, keepdims=True), 1e-12)


def kmeans_task(data, k, seed, pick):
    """One k-means fit (20 starts) on all rows (``pick`` empty) or on the picked rows."""
    rows = data if not len(pick) else data[pick]
    return {'labels': KMeans(n_clusters=k, n_init=KM_STARTS, random_state=seed).fit(rows).labels_}


def choose_k(stage, data):
    """Notebook 56's rule: mean ARI of 20 resamples of 80% per k; the most stable k, ties to the smaller."""
    rng = np.random.default_rng(KM_SEED)
    tasks = {}
    for k in K_RANGE:
        tasks[f'{k}|full'] = (data, k, KM_SEED, np.array([], int))
        for r in range(N_RESAMPLE):
            tasks[f'{k}|{r}'] = (data, k, KM_SEED + r + 1, np.sort(rng.choice(len(data), int(.8 * len(data)), replace=False)))
    runs = cached_tasks(stage, kmeans_task, tasks, params={'k': list(K_RANGE), 'resamples': N_RESAMPLE, 'seed': KM_SEED,
                                                           'starts': KM_STARTS}, inputs={'data': digest(data)})
    stability = {k: float(np.mean([adjusted_rand_score(runs[f'{k}|full'][0]['labels'][tasks[f'{k}|{r}'][3]],
                                                       runs[f'{k}|{r}'][0]['labels']) for r in range(N_RESAMPLE)]))
                 for k in K_RANGE}
    return max(stability, key=lambda k: (round(stability[k], 6), -k)), stability, sum(sec for _, sec in runs.values())


MODULES = {}


def curve_modules(key, number, label, data_of, k_fixed=None, min_per_module=0):
    """Modules of each partition's selected genes, annotated by ORA; calls = pathways enriched in any module."""
    if k_fixed is None:
        k, stability, sec_k = choose_k(f'kselect_{key[:2]}', data_of('species')[1])
        print(f'{key}: mean resample ARI by k', {kk: round(v, 3) for kk, v in stability.items()}, '→ k =', k)
    else:
        k, sec_k = k_fixed, 0.
    prepared = {p: data_of(p) for p in PARTS}
    fit = {p: v for p, v in prepared.items() if len(v[0]) >= max(min_per_module * k, k + 1)}
    runs = cached_tasks(f'modules_{key[:2]}', kmeans_task, {p: (fit[p][1], k, KM_SEED, np.array([], int)) for p in fit},
                        params={'k': k, 'seed': KM_SEED, 'starts': KM_STARTS}, inputs={p: digest(fit[p][1]) for p in fit})
    calls, evidence = {}, None
    t0 = time.perf_counter()
    for p in PARTS:
        if p not in fit:
            calls[p] = []
            print(f'{key} {p}: {len(prepared[p][0])} genes selected, fewer than {min_per_module} × k, so no modules')
            continue
        rows, labels = fit[p][0], runs[p][0]['labels']
        pv = np.column_stack([ora_pvalues(np.isin(np.arange(len(genes)), rows[labels == m]), member_matrix) for m in range(k)])
        q = bh(pv.ravel()).reshape(pv.shape)
        calls[p] = np.array(pathways)[(q <= ALPHA).any(axis=1)]
        if p == 'species':
            evidence = pd.Series(-np.log10(pv.min(axis=1).clip(1e-300)), index=pathways)
            MODULES[key] = {'genes': [list(np.array(genes)[rows[labels == m]]) for m in range(k)], 'q': q,
                            'centroids': np.array([fit[p][1][labels == m].mean(axis=0) for m in range(k)])}
    seconds = sec_k + sum(sec for _, sec in runs.values()) + time.perf_counter() - t0
    register(key, number, label, calls, evidence=evidence, returns=('yes/no', 'shape', 'genes'), runtime=seconds, note=f'k = {k}')


def top_rows(values, share=TOP_SHARE):
    return np.sort(np.argsort(-values)[:int(round(share * len(values)))])


k_b1, _, _ = choose_k('kselect_B1', standardize(curves['species']['delta'][top_rows(curves['species']['T_spatial'])]))
assert k_b1 == K_MODULES, 'notebook 56 chose k = 8'
curve_modules('B1·T10', 5, 'Continuous, gene-first, naive: curve modules of the top 10% T_spatial genes + ORA',
              lambda p: (top_rows(curves[p]['T_spatial']), standardize(curves[p]['delta'][top_rows(curves[p]['T_spatial'])])),
              k_fixed=K_MODULES)
row56 = pd.read_csv(nb56 / 'tables' / 'f_taxonomy.csv').set_index('approach').loc[f'Curve modules (k = {K_MODULES}) + ORA']
assert [len(CALLS['B1·T10'][p]) for p in PARTS] == [int(row56['species calls'])] + [int(x) for x in row56['relabeled calls'].split(', ')]   # Guard: notebook 56
curve_modules('B2·split', 6, 'Continuous, gene-first, specimen-aware: curve modules of the split-plot significant genes + ORA',
              lambda p: (np.flatnonzero(SPLIT_GENES[p]), standardize(curves[p]['delta'][SPLIT_GENES[p]])), min_per_module=10)

## 4 · Comparison

### 4.1 · Programs: counting groups of pathways rather than library entries

Reactome, KEGG and Hallmark contain many near-duplicate pathways, so a call count partly counts
library redundancy. Called pathways are grouped into **programs** by the genes that carry their
signal (notebook 37's rule, made order-invariant):
- a pathway's driver genes are its members in the top 10% of genes by T_spatial, or all members
  when fewer than three qualify;
- pathways are joined by average linkage on 1 − overlap coefficient of driver genes, cut at 0.5.

Each method's calls are grouped on their own for the table. For the overlap figure, the union of
all seven methods' calls is grouped once, so that every method is compared on the same programs.

In [ ]:
from pseudospace.pathway_decoys import overlap_programs

PROGRAM_CUT = .5
contributing = set(gene37.index[gene37.ge(gene37.quantile(1 - TOP_SHARE))])


def driver_programs(ids):
    """Order-invariant programs: notebook 37's rule with pathways sorted by pathway_id before clustering."""
    ids = sorted(set(ids))
    if not ids:
        return pd.Series(dtype=int)
    return overlap_programs(gene_sets, ids, contributing=contributing, cut=PROGRAM_CUT).set_index('pathway_id').program


def program_names(labels, score):
    """Name each program after its member with the highest score (ties by pathway id)."""
    names = {}
    for program, block in labels.groupby(labels):
        best = score.reindex(block.index).fillna(-np.inf).sort_index().sort_values(ascending=False, kind='stable').index[0]
        names[program] = NAMES[best]
    return labels.map(names)


ORDER = [m['key'] for m in METHODS]
TINY = {'A1·GSEA': '1 Whole PT', 'A4·JNT': '2 Discrete', 'A6·JNT': '3 Joint test', 'A6·ORAc': '4 Split-plot ORA', 'A6·GSEA': '7 GSEA',
        'B1·T10': '5 Gene-first naive', 'B2·split': '6 Gene-first aware'}
SHORT = {'A1·GSEA': '1 Whole PT (GSEA)', 'A4·JNT': '2 Discrete S1/S2/S3', 'A6·JNT': '3 Continuous, joint test (primary)',
         'A6·ORAc': '4 Continuous, split-plot list + ORA', 'A6·GSEA': '7 Continuous, GSEA', 'B1·T10': '5 Gene-first, naive',
         'B2·split': '6 Gene-first, specimen-aware'}
c1_drivers = driver_programs(CALLS['A6·JNT']['species'])
union_programs = driver_programs(set().union(*[CALLS[k]['species'] for k in ORDER]))
PROGRAM_SETS = {k: set(union_programs[sorted(CALLS[k]['species'])]) for k in ORDER}
print(f"Union of the seven methods' calls: {len(union_programs)} pathways in {union_programs.nunique()} programs")

### 4.2 · External replication of the specific methods

This is notebook 50's construction, as in notebook 63:
- **References.** Census and Lake/KPMP human cortex donors, each against Census male mice.
- **Statistic.** The signed species × (S3 − early) difference across donors.
- **Score.** The joint replication z of each called pathway, averaged over the method's calls.
- **p.** The share of 999 within-strata gene relabelings that reach that mean z. Each relabeling is
  applied to every pathway at once, so overlaps between called pathways are kept.

The frozen rule runs this only for methods that pass the relabeling rule and have at least five
calls.

In [ ]:
from pseudospace.pathway_decoys import welch_t
from pseudospace.pathway_remodeling import matching_strata
from pseudospace.set_collection_null import JointReplicationZ, stratified_permutation, upper_p

covariates = inputs['covariates']


def census_donors(label, segment_map, early_parts):
    table = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    meta = meta[~meta.development_stage.str.contains('embryo|E16|Theiler', case=False)].copy()
    meta['segment'] = meta.cell_type.map(segment_map)
    meta = meta[meta.segment.notna() & meta.n_cells.ge(MIN_CELLS_EXTERNAL)]
    complete = meta.groupby('donor_id').segment.apply(lambda s: set(segment_map.values()).issubset(set(s)))
    meta = meta[meta.donor_id.isin(complete[complete].index)]
    late, early, info = {}, {}, []

    def log_cpm_of(index):
        return np.log2(table.loc[index].sum() / meta.loc[index, 'total_counts_all_genes'].sum() * 1e6 + 1)

    for (donor, sex), block in meta.groupby(['donor_id', 'sex']):
        late[donor] = log_cpm_of(block.index[block.segment.eq('S3')])
        early[donor] = log_cpm_of(block.index[block.segment.isin(early_parts)])
        info.append({'donor': donor, 'sex': sex})
    frame = lambda values: pd.DataFrame(values).T.reindex(columns=genes)
    return frame(late), frame(early), pd.DataFrame(info).set_index('donor')


h_late, h_early, _ = census_donors('human', {'kidney proximal convoluted tubule epithelial cell': 'early',
                                            'epithelial cell of proximal tubule segment 3': 'S3'}, ['early'])
m_late, m_early, m_info = census_donors('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                                                 'epithelial cell of proximal tubule segment 2': 'S2',
                                                 'epithelial cell of proximal tubule segment 3': 'S3'}, ['S1', 'S2'])
males = m_info.index[m_info.sex.eq('male')]
lake = newest(nb50_cache, 'lake_pseudobulk')          # notebook 50's Lake/KPMP cortex pseudobulk, in our gene order
assert lake['sums'].shape[1] == len(genes)
lake_meta = pd.DataFrame([g.split('|') for g in lake['groups'].astype(str)], columns=['donor', 'sex', 'segment'])
lake_meta['nuclei'] = lake['nuclei']
nuclei = lake_meta.pivot_table(index='donor', columns='segment', values='nuclei', aggfunc='sum', fill_value=0)
lake_donors = nuclei.index[(nuclei.reindex(columns=['S1', 'S2', 'S3'], fill_value=0) >= MIN_CELLS_EXTERNAL).all(axis=1)]
lake_sums = pd.DataFrame(lake['sums'], columns=genes)


def lake_log_cpm(donor, parts):
    rows = lake_meta.index[lake_meta.donor.eq(donor) & lake_meta.segment.isin(parts)]
    return np.log2(lake_sums.loc[rows].sum() / lake['totals'][rows].sum() * 1e6 + 1)


lake_late = pd.DataFrame({d: lake_log_cpm(d, ['S3']) for d in lake_donors}).T
lake_early = pd.DataFrame({d: lake_log_cpm(d, ['S1', 'S2']) for d in lake_donors}).T
short_segment = pd.Series(segment).str.replace('PT-', '', regex=False).to_numpy()
ours_contrast = pd.DataFrame({n: np.asarray(inputs['Y'][(specimen == n) & (short_segment == 'S3')].mean(axis=0)).ravel()
                              - np.asarray(inputs['Y'][(specimen == n) & np.isin(short_segment, ['S1', 'S2'])].mean(axis=0)).ravel()
                              for n in ALL_SPECIMENS}, index=genes)


def positional_reference(human_late, human_early, mouse_late, mouse_early, mouse_donors):
    """Notebook 50's signed species x (S3 - early) reference (Welch t across donors, signed by our cohort)."""
    hd = human_late - human_early
    md = (mouse_late - mouse_early).loc[mouse_donors]
    level_h = pd.concat([human_late.mean(), human_early.mean()], axis=1).max(axis=1)
    level_m = pd.concat([mouse_late.loc[mouse_donors].mean(), mouse_early.loc[mouse_donors].mean()], axis=1).max(axis=1)
    keep = (covariates[['detection_mouse', 'detection_human']].min(axis=1).ge(.10) & level_h.ge(1) & level_m.ge(1)
            & hd.notna().all() & md.notna().all())
    g = keep.index[keep.to_numpy()]
    centred = ours_contrast - ours_contrast.loc[g].median()
    ours_c = centred[SPECIMENS['human']].mean(axis=1) - centred[SPECIMENS['mouse']].mean(axis=1)
    hc, mc = hd.sub(hd[g].median(axis=1), axis=0), md.sub(md[g].median(axis=1), axis=0)
    t = pd.Series(welch_t(hc[g].to_numpy(), mc[g].to_numpy())[0], index=g)
    strata_g = matching_strata(pd.DataFrame({'mean_expression': ((level_h + level_m) / 2).loc[g].to_numpy(),
                                             'detection': covariates.loc[g, ['detection_mouse', 'detection_human']].min(axis=1).to_numpy(),
                                             'coverage': covariates.loc[g, 'coverage'].to_numpy()}), bins=3)
    sets_g = {p: [x for x in v if x in set(g)] for p, v in gene_sets.items()}
    sets_g = {p: v for p, v in sets_g.items() if 5 <= len(v) < len(g)}
    residuals = pd.concat([hc[g] - hc[g].mean(), mc[g] - mc[g].mean()])
    return {'statistic': np.sign(ours_c.loc[g]) * t, 'strata': strata_g, 'sets': sets_g, 'residuals': residuals.to_numpy()}


REFERENCES = {'Census': positional_reference(h_late, h_early, m_late, m_early, males),
              'Lake': positional_reference(lake_late, lake_early, m_late, m_early, males)}
SCORERS = {k: JointReplicationZ(r['statistic'], r['strata'], r['sets'], r['residuals']) for k, r in REFERENCES.items()}
robust36 = table37.index[table37.robust.astype(bool)]
for ref, slug50 in (('Census', 'census_male'), ('Lake', 'lake_male')):   # Guard: notebook 50's replication z
    saved = newest(nb50_cache, f'collection_null_{slug50}_signed_matched')
    theirs = pd.Series(saved['set_z'], index=saved['set_ids'].astype(str))
    mine = pd.Series(SCORERS[ref].z(), index=SCORERS[ref].ids)
    shared_ids = [p for p in robust36 if p in theirs.index]
    np.testing.assert_allclose(mine[shared_ids], theirs[shared_ids], rtol=1e-6, atol=1e-9)

RUN_ID = {'Census': 0, 'Lake': 2}   # notebook 63's seed ids for the two positional references
ELIGIBLE = [m['key'] for m in METHODS if m['specific'] and m['species calls'] >= MIN_CALLS_EXTERNAL]
COLLECTIONS = {}
for ref, scorer in SCORERS.items():
    index = {p: i for i, p in enumerate(scorer.ids)}
    COLLECTIONS[ref] = {k: np.array([index[p] for p in sorted(CALLS[k]['species']) if p in index]) for k in ELIGIBLE}


def replication_chunk(scorer, rows, strata_, seed_ids, n):
    """``n`` within-strata relabelings (seeded by the chunk's ids); mean joint z of every collection."""
    rng = np.random.default_rng(seed_ids)
    draws = np.empty((n, len(rows)))
    for b in range(n):
        z = scorer.z(stratified_permutation(strata_, rng))
        draws[b] = [z[r].mean() if len(r) else np.nan for r in rows]
    return {'draws': draws}


chunk_sizes = [len(c) for c in np.array_split(np.arange(N_PERM_EXTERNAL), EXTERNAL_CHUNKS)]
replication_runs = cached_tasks(
    'external_replication', replication_chunk,
    {f'{ref}|{c}': (SCORERS[ref], list(COLLECTIONS[ref].values()), REFERENCES[ref]['strata'], [EXTERNAL_SEED, RUN_ID[ref], c], n)
     for ref in SCORERS for c, n in enumerate(chunk_sizes)},
    params={'n_perm': N_PERM_EXTERNAL, 'chunks': EXTERNAL_CHUNKS, 'seed': EXTERNAL_SEED},
    inputs={'references': {ref: [digest(r['statistic']), digest(r['residuals']), digest(sorted(r['sets'].items()))]
                           for ref, r in REFERENCES.items()}, 'collections': COLLECTIONS})
replication_rows = []
for ref, scorer in SCORERS.items():
    draws = np.vstack([replication_runs[f'{ref}|{c}'][0]['draws'] for c in range(EXTERNAL_CHUNKS)])
    observed_z = scorer.z()
    for j, (key, rows_) in enumerate(COLLECTIONS[ref].items()):
        observed = float(observed_z[rows_].mean())
        replication_rows.append({'key': key, 'reference': ref, 'calls testable': len(rows_), 'mean z': observed,
                                 'p': upper_p(observed, draws[:, j])})
REPLICATION = pd.DataFrame(replication_rows)
REPLICATION.to_csv(tables / 'external_replication.csv', index=False)

Parallel execution check: the k-means fits and the replication chunks give identical outputs with
one worker and with `N_JOBS` workers.

In [ ]:
check_tasks = {
    'k-means': (kmeans_task, [(standardize(curves['species']['delta'][top_rows(curves['species']['T_spatial'])]), 5, KM_SEED + r,
                               np.sort(np.random.default_rng(r).choice(1107, 885, replace=False))) for r in range(3)]),
    'replication': (replication_chunk, [(SCORERS['Census'], list(COLLECTIONS['Census'].values()), REFERENCES['Census']['strata'],
                                         [EXTERNAL_SEED, 0, c], 3) for c in range(3)])}
for name, (function, args) in check_tasks.items():
    serial, _ = fan_out(function, args, n_jobs=1)
    parallel, _ = fan_out(function, args, n_jobs=N_JOBS)
    assert all(np.array_equal(a[k], b[k], equal_nan=True) for a, b in zip(serial, parallel) for k in a), name
print(f'Parallel check passed: n_jobs = 1 and n_jobs = {N_JOBS} give identical outputs for', ', '.join(check_tasks))

### 4.3 · One table

**Columns:**
- species calls, then the calls under the two relabelings;
- pass or fail for the relabeling rule;
- programs, the method's own calls grouped by the rule above;
- external replication, mean z (p), with – for methods the rule does not replicate;
- **returns**, what a call gives beyond yes or no:
  - **direction:** which species is higher;
  - **location:** where along the PT;
  - **shape:** a positional profile;
  - **genes:** the genes behind the call.

In [ ]:
TABLE = pd.DataFrame(METHODS).set_index('key')
TABLE['programs'] = [driver_programs(CALLS[k]['species']).nunique() for k in ORDER]
for ref in ('Census', 'Lake'):
    block = REPLICATION[REPLICATION.reference.eq(ref)].set_index('key')
    TABLE[f'{ref} z'], TABLE[f'{ref} p'] = block['mean z'].reindex(ORDER), block['p'].reindex(ORDER)
TABLE['relabeling rule'] = np.where(TABLE.specific, 'pass', 'fail')
TABLE['returns'] = [', '.join(r for r in RETURNS if TABLE.loc[k, r]) for k in ORDER]
for ref in ('Census', 'Lake'):
    TABLE[f'{ref} replication'] = [f'{z:.2f} (p {p:.3f})' if pd.notna(z) else '–' for z, p in zip(TABLE[f'{ref} z'], TABLE[f'{ref} p'])]
TABLE.to_csv(tables / 'methods.csv')
COLUMNS = ['#', 'method', 'species calls', 'relabeled calls', 'relabeling rule', 'programs', 'Census replication',
           'Lake replication', 'returns']
display(TABLE[COLUMNS])

### 4.4 · Specificity: species calls against relabeled calls

- **Filled dot:** each method's species calls.
- **Open dots:** the calls under the two specimen relabelings, where no species difference is
  left to find.
- **Dashed line:** 5% of the tested pathways, the most one relabeling may call.

A specific method has a filled dot far to the right of its open dots.

In [ ]:
fig, ax = plt.subplots(figsize=(120 * MM, 52 * MM))
y = np.arange(len(ORDER))[::-1]
for yi, key in zip(y, ORDER):
    species_n = len(CALLS[key]['species'])
    relabeled = [len(CALLS[key][s]) for s in SWAPS]
    ax.plot([min([species_n] + relabeled) + .5, max([species_n] + relabeled) + .5], [yi, yi], color=LIGHT, lw=1, zorder=1)
    ax.scatter([r + .5 for r in relabeled], [yi, yi], s=22, facecolor='white', edgecolor=NULL, lw=1, zorder=2)
    ax.scatter(species_n + .5, yi, s=26, color=INK, zorder=3)
    ax.text(1250, yi, f"{species_n} vs {', '.join(map(str, relabeled))} · {'pass' if TABLE.loc[key, 'specific'] else 'fail'}",
            va='center', fontsize=5.5, color=INK if TABLE.loc[key, 'specific'] else MUTED)
ax.axvline(NULL_RATE * len(pathways) + .5, color=NULL, lw=.7, ls='--')
ax.set_xscale('log')
ax.set_xlim(.4, 1200)
ax.set_xticks([.5, 1.5, 10.5, 100.5, 1000.5], ['0', '1', '10', '100', '1000'])
ax.set_yticks(y, [SHORT[k] for k in ORDER])
ax.set_xlabel('pathways called (log scale)')
ax.scatter([], [], s=26, color=INK, label='species split')
ax.scatter([], [], s=22, facecolor='white', edgecolor=NULL, lw=1, label='specimen relabelings')
ax.plot([], [], color=NULL, lw=.7, ls='--', label='5% of tested pathways')
ax.legend(loc='upper left', bbox_to_anchor=(0, 1.22), ncol=3)
for ext in ('pdf', 'png'):
    fig.savefig(figures / f'fig1_specificity.{ext}', bbox_inches='tight')
plt.show()

### 4.5 · Overlap of the seven methods at program level

The union of all seven methods' species calls is grouped into programs once.

**Left.** The Jaccard index of the programs each pair of methods hits.

**Right.** The largest exact intersections: how many programs are hit by exactly that combination
of methods.

In [ ]:
jac = pd.DataFrame([[len(PROGRAM_SETS[a] & PROGRAM_SETS[b]) / max(len(PROGRAM_SETS[a] | PROGRAM_SETS[b]), 1) for b in ORDER]
                    for a in ORDER], index=ORDER, columns=ORDER)
jac.to_csv(tables / 'program_jaccard.csv')
hits = pd.DataFrame({k: [p in PROGRAM_SETS[k] for p in sorted(union_programs.unique())] for k in ORDER},
                    index=sorted(union_programs.unique()))
combos = hits.value_counts().sort_values(ascending=False)
combos.rename('programs').to_frame().to_csv(tables / 'program_intersections.csv')

fig = plt.figure(figsize=(180 * MM, 70 * MM))
gs = fig.add_gridspec(2, 2, width_ratios=[.8, 1.5], height_ratios=[1.2, 1], wspace=.42, hspace=.06)
ax = fig.add_subplot(gs[:, 0])
image = ax.imshow(jac.to_numpy(), cmap='Greens', vmin=0, vmax=1)
for i in range(len(ORDER)):
    for j in range(len(ORDER)):
        ax.text(j, i, f'{jac.iloc[i, j]:.2f}', ha='center', va='center', fontsize=5, color='white' if jac.iloc[i, j] > .6 else INK)
ax.set_xticks(range(len(ORDER)), [SHORT[k].split(' ')[0] for k in ORDER])
ax.set_yticks(range(len(ORDER)), [f'{SHORT[k]} ({len(PROGRAM_SETS[k])})' for k in ORDER])
ax.set_title('a  Jaccard index of programs hit', loc='left')
top = combos.head(12)
bars = fig.add_subplot(gs[0, 1])
dots = fig.add_subplot(gs[1, 1], sharex=bars)
x = np.arange(len(top))
bars.bar(x, top.to_numpy(), color=INK, width=.6)
for xi, v in zip(x, top.to_numpy()):
    bars.text(xi, v, str(v), ha='center', va='bottom', fontsize=5)
bars.set_ylabel('programs')
bars.set_title(f'b  Exact intersections ({union_programs.nunique()} programs in the union; 12 largest)', loc='left')
plt.setp(bars.get_xticklabels(), visible=False)
for i, key in enumerate(ORDER):
    present = np.array([combo[i] for combo in top.index])
    dots.scatter(x, np.full(len(x), i), s=12, color=np.where(present, INK, LIGHT), zorder=3)
for xi, combo in zip(x, top.index):
    on = [i for i, flag in enumerate(combo) if flag]
    if len(on) > 1:
        dots.plot([xi, xi], [min(on), max(on)], color=INK, lw=1)
dots.set_yticks(range(len(ORDER)), [TINY[k] for k in ORDER])
dots.set_xticks([])
dots.invert_yaxis()
for side in ('left', 'bottom'):
    dots.spines[side].set_visible(False)
for ext in ('pdf', 'png'):
    fig.savefig(figures / f'fig2_program_overlap.{ext}', bbox_inches='tight')
plt.show()
primary_programs = set(c1_drivers.index)
recovered = {k: f"{sum(any(p in CALLS[k]['species'] for p in block.index) for _, block in c1_drivers.groupby(c1_drivers))} of {c1_drivers.nunique()}"
             for k in ORDER}
print('Primary-method programs (A6·JNT calls grouped on their own) with at least one pathway called by each method:', recovered)

### 4.6 · What the continuous method adds over the discrete method

Method 3 (continuous joint test) and method 2 (discrete S1/S2/S3 steps) use the same set test. They
differ only in how position enters: along the coordinate, or as three zones.

The first table lists the pathways that method 3 calls and method 2 does not. For each:
- **step-screen q:** method 2's q-value, showing how close the discrete screen came;
- **called by S1/S2/S3 ORA:** whether the specific S1/S2/S3 method of notebook 63 (DESeq2 per
  segment, then ORA) calls it. If not, no specific discrete method finds the pathway;
- **probe-sensitive:** method 3's call fails notebook 37's equal-probe check. Restricted to genes
  with equal probe counts in both species, the pathway keeps less than 70% of its effect or q
  exceeds 0.10.

The second table lists the pathways method 2 calls and method 3 does not.

The DESeq2 ORA lists computed here (whole PT and S1/S2/S3), together with the S1/S2/S3 GSEA calls,
also feed the worked-example rules in section 5.

In [ ]:
def deseq_ora(frame, scopes):
    """ORA p per pathway, scope and direction on the DESeq2 lists (BH <= 0.05, |log2FC| >= 1); notebook 63's A1/A3·ORAc."""
    columns, labels = [], []
    for scope in scopes:
        rows = frame[frame.scope.eq(scope)].dropna(subset=['stat']).set_index('gene')
        matrix = membership(gene_sets, list(rows.index))
        for direction in (1, -1):
            selected = rows.padj.le(ALPHA).to_numpy() & (direction * rows.log2FoldChange).ge(LFC).to_numpy()
            columns.append(ora_pvalues(selected, matrix))
            labels.append((scope, direction))
    p = pd.DataFrame(np.column_stack(columns), index=pathways, columns=pd.MultiIndex.from_tuples(labels))
    q = bh(p.to_numpy().ravel()).reshape(p.shape)
    return set(p.index[(q <= ALPHA).any(axis=1)]), -np.log10(p.min(axis=1).clip(lower=1e-300))


species_deseq = deseq[deseq.partition.eq('species')]
RULE_CALLS, RULE_EVIDENCE = {}, {}
RULE_CALLS['A1·ORAc'], RULE_EVIDENCE['A1·ORAc'] = deseq_ora(species_deseq, ['whole_PT'])
RULE_CALLS['A3·ORAc'], RULE_EVIDENCE['A3·ORAc'] = deseq_ora(species_deseq, ['PT-S1', 'PT-S2', 'PT-S3'])
segs = gsea31[gsea31.family.eq('segments') & gsea31.partition.eq('species')]
RULE_CALLS['A3·GSEA'] = set(segs.pathway_id[segs.q.le(ALPHA)])
saved56 = pd.read_csv(nb56 / 'tables' / 'f_species_calls_by_approach.csv', index_col=0)
assert RULE_CALLS['A1·ORAc'] == set(saved56.index[saved56['Whole-PT pseudobulk DE + ORA']])     # Guard: notebook 56
assert RULE_CALLS['A3·ORAc'] == set(saved56.index[saved56['S1/S2/S3 pseudobulk DE + ORA']])
assert len(RULE_CALLS['A3·GSEA']) == spec37.loc['2 · S1/S2/S3 pseudobulk + signed GSEA', 'species_calls']   # Guard: notebook 37
probe_pass = table37.probe_equal_pass.reindex(pathways)
q_spatial = spatial37[spatial37.partition.eq('species')].set_index('pathway_id').q_joint
added = sorted(CALLS['A6·JNT']['species'] - CALLS['A4·JNT']['species'])
ADDED = pd.DataFrame({'pathway': NAMES[added], 'method 3 q': q_spatial.reindex(added),
                      'step-screen q (method 2)': steps50.q_joint.reindex(added),
                      'called by S1/S2/S3 ORA': [p in RULE_CALLS['A3·ORAc'] for p in added],
                      'probe-sensitive': ~probe_pass.reindex(added).astype(bool)}).sort_values('step-screen q (method 2)')
ADDED.to_csv(tables / 'continuous_adds_over_discrete.csv')
only_steps = sorted(CALLS['A4·JNT']['species'] - CALLS['A6·JNT']['species'])
STEPS_ONLY = pd.DataFrame({'pathway': NAMES[only_steps], 'step-screen q (method 2)': steps50.q_joint.reindex(only_steps),
                           'method 3 q': q_spatial.reindex(only_steps)})
STEPS_ONLY.to_csv(tables / 'discrete_adds_over_continuous.csv')
no_discrete = ADDED[~ADDED['called by S1/S2/S3 ORA']]
print(f"Method 3 calls {len(added)} pathways that method 2 does not (step-screen q {ADDED['step-screen q (method 2)'].min():.3f}–"
      f"{ADDED['step-screen q (method 2)'].max():.3f}); {len(no_discrete)} of them are called by no specific discrete method "
      f"({int(no_discrete['probe-sensitive'].sum())} probe-sensitive).")
display(ADDED.round(3))
print(f'Method 2 calls {len(only_steps)} pathways that method 3 does not:')
display(STEPS_ONLY.round(3))

## 5 · Worked examples: one pathway, three views

Each row shows one pathway three ways. The value plotted is a display score only, the mean of the
members' species-balanced z-scored expression in each structure; no method above tests it:
- **whole PT:** the specimen means;
- **S1/S2/S3:** specimen × segment means, with bars for the species mean;
- **continuous:** a specimen-balanced species curve (6-df spline), with specimen × 12-bin means as
  points.

The five pathways were chosen by rules fixed in `plan.md` before notebook 61 first ran:
- **E1:** the whole-PT, segment and continuous methods agree. Here it is called by every method
  except method 6.
- **E2:** segment means are flat, but the curves have structure.
- **E3:** a discrete call that has no continuous call.
- **E4:** the direction depends on position.
- **E5:** a step-like pathway, where the curve adds nothing.

Some rules refer to whole-PT and S1/S2/S3 ORA and to S1/S2/S3 GSEA (notebook 63). Those lists come
from section 4.6 and are used here only to apply the rules. Each panel title lists which of the
seven methods call the pathway.

In [ ]:
member_index = {p: pd.Index(genes).get_indexer(v) for p, v in gene_sets.items()}
pathway_delta = pd.DataFrame({p: curves['species']['delta'][idx].mean(axis=0) for p, idx in member_index.items()},
                             index=np.round(grid, 4)).T
evidence = pd.DataFrame({**EVIDENCE, **RULE_EVIDENCE})
C = CALLS['A6·JNT']['species']
scores = newest(nb56 / 'stage_cache', 'structure_scores')

beyond = pd.read_csv(nb57 / 'per_pathway_beyond_steps.csv', index_col=0)
ranks = evidence.rank(pct=True)
rules = {}
agree = sorted(RULE_CALLS['A1·ORAc'] & RULE_CALLS['A3·ORAc'] & CALLS['A4·JNT']['species'] & C)
rules['E1 · all methods agree'] = ranks.loc[agree, ['A1·ORAc', 'A3·ORAc', 'A4·JNT', 'A6·JNT']].min(axis=1).sort_values(ascending=False).index[:1] if agree else []
flat = [p for p in C if steps50.q_joint.get(p, 1) > .05 and p not in RULE_CALLS['A3·GSEA'] and p not in RULE_CALLS['A3·ORAc']
        and bool(beyond.loc[p, 'called test_B_coordinate_steps'])]
rules['E2 · segment means flat, curves structured'] = beyond.loc[flat, 'z_joint test_B_coordinate_steps'].sort_values(ascending=False).index[:1] if flat else []
discrete_only = sorted(RULE_CALLS['A3·ORAc'] - C)
rules['E3 · discrete call, no continuous call'] = RULE_EVIDENCE['A3·ORAc'][discrete_only].sort_values(ascending=False).index[:1] if discrete_only else []
reversing = []
for p in C:
    d = pathway_delta.loc[p].to_numpy()
    threshold = .25 * np.abs(d).max()
    if (d >= threshold).any() and (d <= -threshold).any():
        reversing.append(p)
rules['E4 · direction depends on position'] = EVIDENCE['A6·JNT'][reversing].sort_values(ascending=False).index[:1] if reversing else []
steplike = [p for p in C & CALLS['A4·JNT']['species'] if not bool(beyond.loc[p, 'called test_B_coordinate_steps'])]
rules['E5 · step-like: continuous adds nothing'] = steps50.z_joint[steplike].sort_values(ascending=False).index[:1] if steplike else []
candidates = {'E1': len(agree), 'E2': len(flat), 'E3': len(discrete_only), 'E4': len(reversing), 'E5': len(steplike)}
EXAMPLES = {}
for rule, picked in rules.items():
    if len(picked):
        EXAMPLES[rule] = picked[0]
        print(f'{rule}: {NAMES[picked[0]]} (pathways meeting the rule: {candidates[rule[:2]]})')
    else:
        print(f'{rule}: no pathway meets the rule')

In [ ]:
score_matrix = scores['mean_z']
weights = np.zeros(len(position))
for sp, names_ in SPECIMENS.items():
    for n in names_:
        rows = specimen == n
        weights[rows] = len(position) / (2 * len(names_) * rows.sum())
fine_bins = np.clip(((position - lo) / (hi - lo) * EXAMPLE_BINS).astype(int), 0, EXAMPLE_BINS - 1)
transitions = pd.read_csv(nb37 / 'tables' / 'segment_transitions_by_specimen.csv').groupby('transition').position.mean()


def species_curve(values, species_mask):
    design = make_gam_design(position[species_mask], knots)
    w = np.sqrt(weights[species_mask])
    beta, *_ = np.linalg.lstsq(w[:, None] * design, w * values[species_mask], rcond=None)
    return make_gam_design(grid, knots) @ beta


fig, axes = plt.subplots(len(EXAMPLES), 3, figsize=(180 * MM, 38 * MM * len(EXAMPLES)),
                         gridspec_kw={'width_ratios': [.55, 1, 1.8]}, squeeze=False)
example_rows = []
for r, (rule, pathway) in enumerate(EXAMPLES.items()):
    values = score_matrix[:, pathways.index(pathway)]
    ax_w, ax_s, ax_c = axes[r]
    for k, (sp, color) in enumerate((('mouse', MOUSE), ('human', HUMAN))):
        means = [values[specimen == n].mean() for n in SPECIMENS[sp]]
        ax_w.bar(k, np.mean(means), color=color, alpha=.35, width=.6)
        for n, m in zip(SPECIMENS[sp], means):
            ax_w.scatter(k, m, color=color, marker=SPECIMEN_MARKER[n], s=10, zorder=3, edgecolor='white', linewidth=.4)
        for j, seg in enumerate(('PT-S1', 'PT-S2', 'PT-S3')):
            seg_means = [values[(specimen == n) & (segment == seg)].mean() for n in SPECIMENS[sp]]
            xpos = j + (k - .5) * .38
            ax_s.bar(xpos, np.mean(seg_means), width=.36, color=color, alpha=.35)
            for n, m in zip(SPECIMENS[sp], seg_means):
                ax_s.scatter(xpos, m, color=color, marker=SPECIMEN_MARKER[n], s=10, zorder=3, edgecolor='white', linewidth=.4)
        mask = ~human if sp == 'mouse' else human
        ax_c.plot(grid, species_curve(values, mask), color=color, lw=1.4)
        for n in SPECIMENS[sp]:
            rows = specimen == n
            b = pd.Series(values[rows]).groupby(fine_bins[rows]).mean()
            centres = lo + (b.index.to_numpy() + .5) * (hi - lo) / EXAMPLE_BINS
            ax_c.scatter(centres, b.to_numpy(), color=color, marker=SPECIMEN_MARKER[n], s=7, alpha=.8,
                         edgecolor='white', linewidth=.3, zorder=3)
    for t in transitions:
        ax_c.axvline(t, color=NULL, lw=.6, ls=':')
    ax_w.set_xticks([0, 1], ['mouse', 'human'])
    ax_s.set_xticks([0, 1, 2], ['S1', 'S2', 'S3'])
    ax_c.set_xlim(lo, hi)
    ax_w.set_ylabel('display score (members\' mean z)')
    calls_here = [str(n) for n in sorted(int(TABLE.loc[k, '#']) for k in ORDER if pathway in CALLS[k]['species'])]
    ax_w.set_title(f'{rule.split(" · ")[0]}  whole PT', loc='left')
    ax_s.set_title('S1 / S2 / S3', loc='left')
    ax_c.set_title(f'{NAMES[pathway][:60]} · called by methods {", ".join(calls_here) or "none"}', loc='left')
    if r == len(EXAMPLES) - 1:
        ax_c.set_xlabel('PT coordinate (SCF13); dotted lines: mean S1→S2 and S2→S3 transitions')
    example_rows.append({'example': rule, 'pathway': NAMES[pathway], 'called by methods': ', '.join(calls_here),
                         'step-screen q (A4·JNT)': steps50.q_joint.get(pathway, np.nan),
                         'A6·JNT q_joint': tests37[tests37.statistic.eq('T_spatial') & tests37.partition.eq('species')
                                               & tests37.pathway_id.eq(pathway)].q_joint.iloc[0],
                         'beyond coordinate steps called (notebook 57)': bool(beyond.loc[pathway, 'called test_B_coordinate_steps'])})
handles = [plt.Line2D([], [], color=MOUSE, lw=1.4, label='mouse'), plt.Line2D([], [], color=HUMAN, lw=1.4, label='human'),
           plt.Line2D([], [], color=INK, marker='o', lw=0, markersize=3, label='specimen 1'),
           plt.Line2D([], [], color=INK, marker='s', lw=0, markersize=3, label='specimen 2')]
axes[0, 2].legend(handles=handles, loc='upper left', ncol=4, bbox_to_anchor=(0, 1.32))
fig.tight_layout()
for ext in ('pdf', 'png'):
    fig.savefig(figures / f'fig3_worked_examples.{ext}', bbox_inches='tight')
plt.show()
EXAMPLE_TABLE = pd.DataFrame(example_rows)
EXAMPLE_TABLE.to_csv(tables / 'worked_examples.csv', index=False)
display(EXAMPLE_TABLE.round(4))

## 6 · Gene heatmaps: which genes drive the signal, and where

A pathway score averages its members, so it hides the genes that drive a call and cancels members
that change in opposite directions. These heatmaps show the genes themselves.

- **Rows.** At most 40 genes, chosen by T_spatial. A bullet marks genes in the top 10% by
  T_spatial.
- **Left matrix.** Which member pathway, numbered in the title, contains each gene.
- **Columns.** The 12 coordinate bins, mouse and human side by side.
  - Each gene's specimen-averaged bin means are z-scored within each species, which removes the
    level offset.
  - Dotted lines mark each species' mean S1→S2 and S2→S3 transitions; a dot marks the peak bin.
- **Row order.** By mouse peak bin, then human peak bin.
- **Right of the heatmap.**
  - The notebook 43 zonation class.
  - log10 T_spatial.
  - The whole-PT DESeq2 log2 human/mouse (H/M), so the offset is not lost.
- **Below.** Each specimen's own z-scored bins in the same order, the replication check.

**Three heatmaps:**
1. **E1, where the methods agree.** The primary-method program that contains E1.
2. **E4, where the continuous method shows a sign change the whole-PT method misses.** The
   program that contains E4. Method 1 does not call it.
3. **The gene-first S3 module: method 6's module 3.** This is the curve module of split-plot
   significant genes whose species difference falls toward S3, so mouse is higher there. Its 40
   highest-T_spatial genes are shown, and the left matrix gives its enriched pathways (at most five).
   The species-blind co-expression screen of notebook 63 finds an overlapping 46-gene S3 module that
   shares 31 genes with this one.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

MAX_ROWS, Z_CLIP = 40, 2.5
HEAT = LinearSegmentedColormap.from_list('teal_grey_brown', ['#01665E', '#5AB4AC', '#E6E6E6', '#D8B365', '#8C510A'])
CLASS_COLORS = {'conserved': '#009E73', 'reversal': '#CC79A7', 'mouse-only': MOUSE, 'human-only': HUMAN,
                'indeterminate': '#E3E3E3', 'neither': '#B8B8B8'}
SHORT_SPECIMEN = {'Ctrl1A2': 'M1', 'Ctrl1A4': 'M2', 'HUK1_COR1': 'H1', 'HUK1_MED1': 'H2'}


def specimen_bin_means():
    """Mean log-normalised expression per specimen x coordinate bin (genes x bins per specimen)."""
    out = {}
    for n in ALL_SPECIMENS:
        rows = np.flatnonzero(specimen == n)
        indicator = sparse.csr_matrix((np.ones(len(rows)), (fine_bins[rows], np.arange(len(rows)))),
                                      shape=(EXAMPLE_BINS, len(rows)))
        sums = np.asarray((indicator @ inputs['Y'][rows]).todense())
        out[n] = (sums / np.asarray(indicator.sum(axis=1))).T
    return out


bin_means = cached_payload('specimen_bin_means', specimen_bin_means, root=cache, params={'bins': EXAMPLE_BINS},
                           inputs=input_key, code=code)
species_means = {sp: np.mean([bin_means[n] for n in SPECIMENS[sp]], axis=0) for sp in SPECIMENS}


def zscore_rows(values):
    centred = values - values.mean(axis=1, keepdims=True)
    sd = values.std(axis=1, keepdims=True)
    return np.where(sd > 1e-9, centred / np.where(sd > 1e-9, sd, 1), 0.)


species_z = {sp: zscore_rows(v) for sp, v in species_means.items()}
specimen_z = {n: zscore_rows(v) for n, v in bin_means.items()}
gene_position = pd.Series(np.arange(len(genes)), index=genes)
t_spatial = gene37
top_decile = set(t_spatial.nlargest(int(round(TOP_SHARE * len(genes)))).index)
whole_lfc = deseq[deseq.partition.eq('species') & deseq.scope.eq('whole_PT')].set_index('gene').log2FoldChange
zonation = pd.read_csv(results_root / 'pt_revision_classes' / 'reviewed' / 'tables' / 'gene_classes_symmetric.csv',
                       index_col=0)['union S2-S1 + S3c-early']
species_transitions = pd.read_csv(nb37 / 'tables' / 'segment_transitions_by_specimen.csv').groupby(
    ['species', 'transition']).position.mean()
bin_edges = np.linspace(lo, hi, EXAMPLE_BINS + 1)


def program_rows(member_pathways):
    """Union of the members' contributing genes (fallback: all members), capped at the top 40 by T_spatial."""
    union = []
    for p in member_pathways:
        members = [g for g in gene_sets[p] if g in gene_position.index]
        chosen = [g for g in members if g in contributing]
        union.extend(chosen if len(chosen) >= 3 else members)
    union = sorted(set(union))
    shown = union if len(union) <= MAX_ROWS else list(t_spatial[union].nlargest(MAX_ROWS).index)
    idx = gene_position[shown].to_numpy()
    peaks = pd.DataFrame({'mouse': species_z['mouse'][idx].argmax(axis=1), 'human': species_z['human'][idx].argmax(axis=1),
                          'gene': shown})
    order = peaks.sort_values(['mouse', 'human', 'gene']).index.to_numpy()
    return [shown[i] for i in order], len(union)


def draw_program_heatmap(member_pathways, title, rows=None):
    """Gene heatmap of a program (rows from its members' driver genes) or of given genes (``rows``, e.g. a module)."""
    member_pathways = sorted(member_pathways, key=lambda p: -EVIDENCE['A6·JNT'].get(p, 0))
    if rows is None:
        rows, n_union = program_rows(member_pathways)
    else:
        n_union = len(rows)
    idx = gene_position[rows].to_numpy()
    n, k = len(rows), len(member_pathways)
    height_mm = 40 + n * 2.4 * 1.6 + 3 * k
    fig = plt.figure(figsize=(180 * MM, height_mm * MM))
    widths = [max(k, 1) * .9, 12, 1.2, 12, 1.2, 2.4, 2.4]
    outer = fig.add_gridspec(2, 1, height_ratios=[1, .6], hspace=.16, top=1 - (14 + 3 * k) / height_mm, bottom=.04,
                             left=.12, right=.98)
    top = outer[0].subgridspec(1, 7, width_ratios=widths, wspace=.06)
    bottom = outer[1].subgridspec(1, 7, width_ratios=widths, wspace=.06)
    mx = fig.add_subplot(top[0, 0])
    matrix = np.array([[g in set(gene_sets[p]) for p in member_pathways] for g in rows], float)
    yy, xx = np.nonzero(matrix)
    mx.scatter(xx, yy, s=4, color=INK, linewidths=0)
    mx.set_xlim(-.5, max(k, 1) - .5)
    mx.set_ylim(n - .5, -.5)
    mx.set_xticks(range(k), [str(i + 1) for i in range(k)], fontsize=5)
    mx.xaxis.tick_top()
    mx.set_yticks(range(n), [('• ' if g in top_decile else '') + g for g in rows], fontsize=5)
    for side in ('right', 'bottom'):
        mx.spines[side].set_visible(False)
    mx.set_title('in pathway' if k else '', loc='left', fontsize=5.5, pad=8)
    for col, sp in ((1, 'mouse'), (3, 'human')):
        ax = fig.add_subplot(top[0, col])
        image = ax.imshow(np.clip(species_z[sp][idx], -Z_CLIP, Z_CLIP), cmap=HEAT, vmin=-Z_CLIP, vmax=Z_CLIP,
                          aspect='auto', interpolation='nearest')
        ax.scatter(species_z[sp][idx].argmax(axis=1), np.arange(n), s=3, color=INK, zorder=3, linewidths=0)
        for t in ('S1→S2', 'S2→S3'):
            ax.axvline((species_transitions[(sp, t)] - lo) / (hi - lo) * EXAMPLE_BINS - .5, color=INK, lw=.6, ls=':')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f'{sp}: specimen-averaged, z within species', loc='left', color=MOUSE if sp == 'mouse' else HUMAN)
        sub = bottom[0, col].subgridspec(1, 2, wspace=.08)
        for j, name in enumerate(SPECIMENS[sp]):
            bx = fig.add_subplot(sub[0, j])
            bx.imshow(np.clip(specimen_z[name][idx], -Z_CLIP, Z_CLIP), cmap=HEAT, vmin=-Z_CLIP, vmax=Z_CLIP,
                      aspect='auto', interpolation='nearest')
            for t in ('S1→S2', 'S2→S3'):
                bx.axvline((species_transitions[(sp, t)] - lo) / (hi - lo) * EXAMPLE_BINS - .5, color=INK, lw=.5, ls=':')
            bx.set_xticks([0, EXAMPLE_BINS - 1], ['start', 'end'], fontsize=5)
            bx.set_yticks([])
            bx.set_title(f'{SHORT_SPECIMEN[name]} ({name})', loc='left', fontsize=5.5, fontweight='normal')
    cx = fig.add_subplot(top[0, 4])
    for i, g in enumerate(rows):
        c = zonation.get(g, np.nan)
        cx.add_patch(plt.Rectangle((0, i - .4), 1, .8, color=CLASS_COLORS.get(c, '#FFFFFF'),
                                   ec=LIGHT if c not in CLASS_COLORS else 'none', lw=.3))
    cx.set_xlim(0, 1)
    cx.set_ylim(n - .5, -.5)
    cx.axis('off')
    cx.set_title('class', loc='left', fontsize=5.5)
    tx = fig.add_subplot(top[0, 5])
    tx.barh(np.arange(n), np.log10(t_spatial[rows].clip(lower=1)), color=ACCENT, height=.7)
    tx.set_ylim(n - .5, -.5)
    tx.set_yticks([])
    tx.set_title('log10 T', loc='left', fontsize=5.5)
    tx.tick_params(labelsize=5)
    lx = fig.add_subplot(top[0, 6])
    lfc = whole_lfc.reindex(rows).clip(-6, 6).fillna(0).to_numpy()
    lx.barh(np.arange(n), lfc, color=np.where(lfc > 0, HUMAN, MOUSE), height=.7)
    lx.axvline(0, color=INK, lw=.5)
    lx.set_xlim(-6.5, 6.5)
    lx.set_ylim(n - .5, -.5)
    lx.set_yticks([])
    lx.set_title('log2 H/M', loc='left', fontsize=5.5)
    lx.tick_params(labelsize=5)
    cax = fig.add_subplot(bottom[0, 4:])
    cax.axis('off')
    bar = fig.colorbar(image, ax=cax, orientation='horizontal', fraction=.45, aspect=10, pad=0)
    bar.set_label('z within species (clipped ±2.5)', fontsize=5)
    bar.ax.tick_params(labelsize=5)
    legend = [plt.Rectangle((0, 0), 1, 1, color=v) for v in CLASS_COLORS.values()] + \
        [plt.Rectangle((0, 0), 1, 1, facecolor='white', edgecolor=LIGHT)]
    cax.legend(legend, list(CLASS_COLORS) + ['not classified'], loc='upper left', fontsize=4.5, ncol=2,
               handlelength=1, borderaxespad=0)
    called = [str(n) for n in sorted(int(TABLE.loc[m, '#']) for m in ORDER if any(p in CALLS[m]['species'] for p in member_pathways))]
    header = f'{title} · {n} of {n_union} genes shown' + (f' · members called by methods {", ".join(called) or "none"}' if k else '')
    listing = '; '.join(f'{i + 1} {NAMES[p]}' for i, p in enumerate(member_pathways))
    fig.text(.01, 1 - 4 / height_mm, header, ha='left', va='top', fontsize=6.5, fontweight='bold')
    fig.text(.01, 1 - 8 / height_mm, listing, ha='left', va='top', fontsize=5, wrap=True, color=MUTED)
    return fig


def program_of(pathway, labels):
    return sorted(labels.index[labels.eq(labels[pathway])]) if pathway in labels.index else [pathway]


c1_names = program_names(c1_drivers, EVIDENCE['A6·JNT'])
assert EXAMPLES['E4 · direction depends on position'] not in CALLS['A1·GSEA']['species'], 'E4 should be missed by method 1'
for rule in ('E1 · all methods agree', 'E4 · direction depends on position'):
    pathway = EXAMPLES[rule]
    members = program_of(pathway, c1_drivers)
    fig = draw_program_heatmap(members, f'{rule.split(" · ")[0]} · program "{c1_names[pathway]}" ({len(members)} pathways among the primary-method calls)')
    for ext in ('pdf', 'png'):
        fig.savefig(figures / f'fig4_gene_heatmap_{rule.split(" · ")[0]}.{ext}')
    plt.show()

In [ ]:
module3 = MODULES['B2·split']['genes'][2]
assert {'Slc22a7', 'Slc5a8', 'Cyp7b1'} <= set(module3), "method 6's module 3 should hold the S3 genes"
module3_q = MODULES['B2·split']['q'][:, 2]
enriched3 = [pathways[i] for i in np.argsort(module3_q)[:5] if module3_q[i] <= ALPHA]
top_genes = list(t_spatial[module3].nlargest(MAX_ROWS).index)
idx = gene_position[top_genes].to_numpy()
order_rows = pd.DataFrame({'mouse': species_z['mouse'][idx].argmax(axis=1), 'human': species_z['human'][idx].argmax(axis=1),
                           'gene': top_genes}).sort_values(['mouse', 'human', 'gene']).gene.tolist()
print(f"Method 6 module 3: {len(module3)} genes; {int((module3_q <= ALPHA).sum())} enriched pathways")
fig = draw_program_heatmap(enriched3, f'Method 6 module 3: {len(module3)} split-plot genes, difference falling toward S3; '
                                      f'top {MAX_ROWS} by T_spatial', rows=order_rows)
for ext in ('pdf', 'png'):
    fig.savefig(figures / f'fig4_gene_heatmap_S3_module.{ext}')
plt.show()

## 7 · Summary

1. Whole-PT GSEA (method 1), the usual analysis, is not specific: it calls 158 pathways for species
   but 193 and 255 when the specimens are relabeled.
2. The discrete step screen (2), the continuous joint test (3) and ORA on the split-plot list (4)
   pass the relabeling rule, replicate in both atlases (mean z 1.2–1.5, p 0.001), and each collapse
   to 25 programs.
3. GSEA on the same T_spatial ranking (7) passes but calls about 20% of pathways and replicates only
   in Lake, because it lacks the joint test's expression-matched background and correlation (VIF)
   correction.
4. Gene-first modules fail when built from the top-10% T_spatial genes (5). Built from the split-plot
   list (6), they pass and replicate, but recover only 7 of the primary method's 25 programs.
5. The continuous coordinate adds pathways over S1/S2/S3 zones. Method 3 calls 13 pathways that the
   step screen (2) does not, 7 of them called by no specific discrete method, and the step screen
   calls 3 that method 3 does not. The 13 are all near misses for the step screen (q 0.058–0.136),
   so the coordinate gains where the zones see the difference only weakly. It also shows where
   inside a segment the species differ (E2) and where the direction changes along the PT (E4).
6. Gene-first adds a gene-level view of shapes, such as method 6's module of genes whose species
   difference falls toward S3.

Whole-PT ORA on DESeq2 lists also passes the relabeling rule (80 calls; 0 and 1 relabeled), but it
tests average offsets only. Notebook 63 holds the full grid of 48 methods behind the supplementary
figure.

## 8 · Run record

In [ ]:
record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'n_structures': int(len(position)), 'n_genes': len(genes),
          'n_pathways': len(pathways), 'n_jobs': N_JOBS, 'methods': TABLE.reset_index().to_dict(orient='records'),
          'examples': {r: NAMES[p] for r, p in EXAMPLES.items()}, 'wall_seconds': time.perf_counter() - NOTEBOOK_START}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))
print(f"Notebook wall time {record['wall_seconds'] / 60:.1f} min with {N_JOBS} workers")